# StudySignal: Statistical Inference for Student Performance
This notebook builds a reproducible statistical investigation using a generated educational dataset. The data are synthetic and are intended for demonstrating statistical methods, not for making claims about real students.

**Research questions**
1. Is study time associated with marks?
2. Does average performance differ between group-study and solo-study students?
3. Do mean marks differ across study methods?
4. Is attendance category associated with pass status?
5. What is a 95% confidence interval for the mean mark?
6. How well can a simple regression model estimate marks from selected variables?

Significance level: **α = 0.05**. Statistical significance does not establish causation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 20261028
rng = np.random.default_rng(SEED)
n = 300

study_hours = np.clip(rng.normal(3.8, 1.5, n), 0.5, 9)
sleep_hours = np.clip(rng.normal(6.8, 1.0, n), 3.5, 9.5)
attendance = np.clip(rng.normal(78, 12, n), 35, 100)
screen_time = np.clip(rng.normal(4.5, 1.7, n), 0.5, 10)
stress = np.clip(np.rint(rng.normal(5.5, 2.0, n)), 1, 10)
study_method = rng.choice(["Solo Study", "Group Study", "Online Resources"], n, p=[0.42, 0.30, 0.28])
method_effect = pd.Series(study_method).map({"Solo Study": 0, "Group Study": 2.8, "Online Resources": 1.3}).to_numpy()
marks = 28 + 4.2*study_hours + 0.22*attendance + 1.6*sleep_hours - 1.15*stress - 0.8*screen_time + method_effect + rng.normal(0, 7.5, n)
marks = np.clip(marks, 0, 100)

df = pd.DataFrame({
    "Student_ID": [f"STU{i:03d}" for i in range(1, n+1)],
    "Study_Hours": np.round(study_hours, 2),
    "Sleep_Hours": np.round(sleep_hours, 2),
    "Attendance_Percent": np.round(attendance, 1),
    "Screen_Time_Hours": np.round(screen_time, 2),
    "Stress_Score": stress.astype(int),
    "Study_Method": study_method,
    "Group_Study": np.where(study_method == "Group Study", "Yes", "No"),
    "Marks": np.round(marks, 1)
})
df["Pass_Status"] = np.where(df["Marks"] >= 40, "Pass", "Needs Support")
df.to_csv("studysignal_student_dataset.csv", index=False)
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDescriptive statistics:")
display(df.describe(include="all").T)
print("\nPass status distribution:")
display(df["Pass_Status"].value_counts())

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df["Study_Hours"], df["Marks"], alpha=0.65)
slope, intercept = np.polyfit(df["Study_Hours"], df["Marks"], 1)
x_line = np.linspace(df["Study_Hours"].min(), df["Study_Hours"].max(), 100)
plt.plot(x_line, slope*x_line + intercept)
plt.xlabel("Study hours per day")
plt.ylabel("Marks (%)")
plt.title("Study hours and academic marks")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
df.boxplot(column="Marks", by="Study_Method", grid=False)
plt.suptitle("")
plt.title("Marks by study method")
plt.xlabel("Study method")
plt.ylabel("Marks (%)")
plt.tight_layout()
plt.show()

In [ ]:
alpha = 0.05

pearson_r, pearson_p = stats.pearsonr(df["Study_Hours"], df["Marks"])
spearman_r, spearman_p = stats.spearmanr(df["Study_Hours"], df["Marks"])
print(f"Pearson r = {pearson_r:.4f}, p-value = {pearson_p:.6g}")
print(f"Spearman rho = {spearman_r:.4f}, p-value = {spearman_p:.6g}")
print("Pearson decision:", "Reject H0" if pearson_p < alpha else "Fail to reject H0")
print("Interpretation: this tests association, not causation.")

In [ ]:
group = df.loc[df["Study_Method"] == "Group Study", "Marks"]
solo = df.loc[df["Study_Method"] == "Solo Study", "Marks"]
t_stat, t_p = stats.ttest_ind(group, solo, equal_var=False)
print("Group-study mean:", round(group.mean(), 2))
print("Solo-study mean:", round(solo.mean(), 2))
print(f"Welch t-statistic = {t_stat:.4f}, p-value = {t_p:.6g}")
print("Decision:", "Reject H0" if t_p < alpha else "Fail to reject H0")

In [ ]:
solo = df.loc[df["Study_Method"] == "Solo Study", "Marks"]
group = df.loc[df["Study_Method"] == "Group Study", "Marks"]
online = df.loc[df["Study_Method"] == "Online Resources", "Marks"]
f_stat, anova_p = stats.f_oneway(solo, group, online)
print("Group means:")
print(df.groupby("Study_Method")["Marks"].mean().round(2))
print(f"One-way ANOVA F = {f_stat:.4f}, p-value = {anova_p:.6g}")
print("Decision:", "Reject H0" if anova_p < alpha else "Fail to reject H0")
print("If significant, follow up with an appropriate post-hoc test.")

In [ ]:
attendance_band = pd.cut(
    df["Attendance_Percent"],
    bins=[0, 60, 80, 101],
    labels=["Low (<60)", "Medium (60–80)", "High (>80)"],
    right=False
)
contingency = pd.crosstab(attendance_band, df["Pass_Status"])
chi2, chi_p, dof, expected = stats.chi2_contingency(contingency)
print("Observed contingency table:")
display(contingency)
print(f"Chi-square = {chi2:.4f}, df = {dof}, p-value = {chi_p:.6g}")
print("Decision:", "Reject H0" if chi_p < alpha else "Fail to reject H0")

In [ ]:
mean_marks = df["Marks"].mean()
ci_low, ci_high = stats.t.interval(
    confidence=0.95,
    df=len(df["Marks"]) - 1,
    loc=mean_marks,
    scale=stats.sem(df["Marks"])
)
print(f"Mean marks = {mean_marks:.2f}")
print(f"95% confidence interval = ({ci_low:.2f}, {ci_high:.2f})")

In [ ]:
features = ["Study_Hours", "Sleep_Hours", "Attendance_Percent", "Screen_Time_Hours", "Stress_Score"]
X = df[features]
y = df["Marks"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")
print("\nCoefficients:")
display(pd.DataFrame({"Feature": features, "Coefficient": model.coef_}).sort_values("Coefficient", ascending=False))

In [ ]:
summary = pd.DataFrame([
    ["Pearson correlation: study hours vs marks", pearson_r, pearson_p, "Association"],
    ["Welch t-test: group vs solo", t_stat, t_p, "Difference in two means"],
    ["One-way ANOVA: study methods", f_stat, anova_p, "Difference across three means"],
    ["Chi-square: attendance band vs pass status", chi2, chi_p, "Association between categories"]
], columns=["Analysis", "Test statistic", "p-value", "Purpose"])
display(summary)
print("All records in this notebook are synthetic. Do not interpret results as evidence about an actual college population.")